# Data Ingestion

Ingesting noaa weather data into bronze datalake in s3.

In [ ]:
# Dump data into s3
!aws s3 sync s3://noaa-ghcn-pds/parquet/by_year/ \
            s3://aebel-datalake/bronze/noaa_ghcn_daily/ \
            --exclude "*" --include "YEAR=20*" \
            # --dryrun 

In [ ]:
!aws s3 ls s3://noaa-ghcn-pds/

In [ ]:
!aws s3 cp s3://noaa-ghcn-pds/readme.txt -

In [ ]:
!aws s3 ls s3://aebel-datalake/bronze/noaa_ghcn_daily/YEAR=2001/ELEMENT=PRCP/

## Investigation

In [ ]:
import polars as pl
lf = pl.scan_parquet("s3://aebel-datalake/bronze/noaa_ghcn_daily/YEAR=2001/")
lf.collect_schema()

In [ ]:
lf.head().collect()

# Athena


In [ ]:
import os
from sqlalchemy import create_engine

os.environ["AWS_PROFILE"] = "default"      # or rely on env vars / SSO / instance role

REGION    = "eu-west-1"
WORKGROUP = "primary"
STAGING   = "s3://aebel-athena-results/results/"

engine = create_engine(
    f"awsathena+arrow://@athena.{REGION}.amazonaws.com:443"
    f"?s3_staging_dir={STAGING}&work_group={WORKGROUP}"
)

%load_ext sql
%config SqlMagic.autopolars = True         # results come back as polars.DataFrame
%config SqlMagic.displaylimit = 20
%config SqlMagic.feedback = 0
%sql engine

In [ ]:
%%sql
describe datalake_bronze.noaa_ghcn_daily

In [ ]:
%%sql
SELECT id, data_value / 10.0 AS precip_mm, q_flag
FROM datalake_bronze.noaa_ghcn_daily
WHERE year = 2001
  AND element = 'PRCP'
  AND "date" = '20010705'
  AND id LIKE 'IN%'
ORDER BY precip_mm DESC;